# LoRA Rank x Exposure Ablation (run 2)

**Run 1 was uninformative, and this run fixes it.** Varying rank alone at 30 epochs
returned 0.995 at rank 4 and 1.000 at every larger rank - a ceiling. The exposure
ablation in Section 5.6.1 had already shown that memorization saturates at 16
exposures per record, and run 1 used 60, so every cell sat deep in the saturated
regime and the probe had no room to discriminate. That is a measurement problem,
not a negative result.

This run moves the operating point to where the outcome can still move, and sweeps
**rank against exposure** rather than rank alone.

| Reading | What it means for the thesis |
|---|---|
| Rate rises with rank in the unsaturated rows | Adapter-capacity explanation **confirmed**; 5.6 upgrades from hypothesis to evidence |
| Rows flat wherever they are not saturated | Explanation **refuted**; 5.6 withdraws the mechanism and reports the capacity question as open |
| Saturation threshold shifts right as rank falls | The sharper confirmation - a smaller adapter needs more exposures to hold the same corpus |

Send the JSON back either way. A refutation rewrites the chapter; it is not a wasted run.

---

**Before starting:** Runtime -> Change runtime type -> **L4 GPU**.

Every grid cell checkpoints to Drive, so a disconnect costs at most the cell in flight.

## 1 - Confirm the GPU

In [ ]:
!nvidia-smi

## 2 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/piibench'
os.makedirs(DRIVE, exist_ok=True)
print('results will be written to:', DRIVE)

## 3 - Unpack the bundle

Upload `piibench_rank_ablation.zip` into the **`piibench` folder** of your Drive first (it replaces the run-1 zip if that is still there), then run this cell.

In [ ]:
import zipfile, os
src = f'{DRIVE}/piibench_rank_ablation.zip'
assert os.path.exists(src), f'not found: {src} - upload the zip to Drive first'
zipfile.ZipFile(src).extractall('/content/work')
os.chdir('/content/work')
print(sorted(os.listdir('.')))

## 4 - Install dependencies
(about 2 minutes)

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pandas matplotlib

## 5 - Run the grid: 5 ranks x 5 exposure levels

40 records; exposures 2 / 4 / 6 / 8 / 12 per record. These bracket the knee of the
exposure curve, which sits near 8 exposures where the rate is about 0.60.

**About 75 minutes on an L4.** Twenty-five cells, but each trains far fewer epochs
than run 1, so the total is close to what run 1 cost - and this time it carries
information.

Adapters go to local scratch; only the checkpoint JSON and the figure reach Drive.

In [ ]:
!python ablation_rank.py \
  --model Qwen/Qwen2.5-1.5B \
  --persons 40 \
  --ranks 4 8 16 32 64 \
  --epochs-list 1 2 3 4 6 \
  --out-dir $DRIVE/rank_grid_40 \
  --adapter-dir /content/adapters_grid

## 6 - Read the grid

Cell 5 prints this too; this re-prints it without recomputing anything and shows the figure.

In [ ]:
import json, os
from IPython.display import Image, display
rows = json.load(open(f'{DRIVE}/rank_grid_40/ablation_rank.json'))
ranks = sorted({r['rank'] for r in rows})
print('  exposures | ' + ' '.join(f'r={r:<4}' for r in ranks))
informative = []
for ex in sorted({r['exposures'] for r in rows}):
    cells = {r['rank']: r['mem_rate'] for r in rows if r['exposures'] == ex}
    line = ' '.join(f"{cells.get(r, float('nan')):<6.3f}" for r in ranks)
    vals = list(cells.values())
    if min(vals) >= 0.98:
        note = '  <- ceiling'
    elif max(vals) <= 0.02:
        note = '  <- floor'
    else:
        informative.append((ex, cells))
        note = f'  <- INFORMATIVE (spread {max(vals) - min(vals):+.3f})'
    print(f'  {ex:>9} | {line}{note}')
print()
if not informative:
    print('  No informative row. Lower --epochs-list further and re-run cell 5;')
    print('  completed cells are skipped, so nothing is recomputed.')
else:
    rise = sum(1 for ex, c in informative if c[max(c)] - c[min(c)] > 0.10)
    print(f'  {len(informative)} informative row(s), {rise} rising with rank.')
    if rise:
        print('  RISING -> supports the adapter-capacity explanation of section 5.6.')
    else:
        print('  FLAT   -> refutes it; section 5.6 gets rewritten. Still a result.')
fig = f'{DRIVE}/rank_grid_40/fig_ablation_rank.png'
if os.path.exists(fig):
    display(Image(fig))

## 7 - Only if cell 6 found no informative row

Adds one higher exposure level. Completed cells are skipped, so nothing already measured is recomputed.

In [ ]:
!python ablation_rank.py \
  --model Qwen/Qwen2.5-1.5B \
  --persons 40 \
  --ranks 4 8 16 32 64 \
  --epochs-list 1 2 3 4 6 8 \
  --out-dir $DRIVE/rank_grid_40 \
  --adapter-dir /content/adapters_grid

## 8 - Collect the results

A few hundred kilobytes: JSON and figures only.

In [ ]:
import shutil, os, glob
os.makedirs('/content/send', exist_ok=True)
for d in ['rank_grid_40', 'rank_ablation_40', 'rank_ablation_140']:
    src = f'{DRIVE}/{d}'
    if not os.path.isdir(src):
        continue
    dst = f'/content/send/{d}'
    os.makedirs(dst, exist_ok=True)
    for pat in ['ablation_rank.json', 'fig_ablation_rank.png', 'fig_ablation_rank.pdf']:
        for f in glob.glob(f'{src}/{pat}'):
            shutil.copy(f, dst)
shutil.make_archive(f'{DRIVE}/rank_grid_results', 'zip', '/content/send')
sz = os.path.getsize(f'{DRIVE}/rank_grid_results.zip')
print('wrote', f'{DRIVE}/rank_grid_results.zip', round(sz / 1024, 1), 'KB')
print('Download it from Drive and send it back.')

---
### If the session drops
Re-run cells 2 -> 3 -> 4, then cell 5. You should see `resuming: N grid cells already measured`.

### If you hit CUDA out of memory
Rank 64 is the heaviest. Drop it: `--ranks 4 8 16 32`. Four ranks still show the shape.

### What good output looks like
Each cell prints `[rank R, E exposures] fine-tuning ...` then `-> verbatim memorization rate = 0.xxxx [checkpointed]`. You want rates spread across the range, not all 0.000 and not all 1.000.